In [ ]:
# DASHBOARD NOTEBOOK

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import json
import runpy
import re
import gc

# GENERAL PARAMETERS
RAW_ROOT =  Path("data/raw_signal")
RAW_EMG_DIR   = RAW_ROOT / "emg"
RAW_BIA_DIR   = RAW_ROOT / "bia"
RAW_NIRS_DIR  = RAW_ROOT / "nirs"
RAW_MYOTON_DIR = RAW_ROOT / "myoton"

## LABELING PARAMETERS
# PROTOCOLE SEQUENCE ORDER 
SEQ_ORDER = "INIT WU REST0 MVC_REF REST1 SVC_REF REST2 EX_DYN REST3 MVC_RECOV_DYN REST4 EX_STA REST5 MVC_RECOV_STA END".split()




## 0. SUBJECT SELECTION

In [ ]:

# RUN_ID = "011BeSa_20251023" # reexport- missing seq in master
# RUN_ID = "081PaFe_20251113" # reexport -missing seq in master
# RUN_ID = "487LePa_20251003" 
# RUN_ID = "092MaLe_20251031" 
# RUN_ID = "101SaCl_20251127" # OK
RUN_ID = "133DeMa_20251021" 
# RUN_ID = "197LaAl_20251016" 
# RUN_ID = "209CeJa_20251105" 
# RUN_ID = "222SoMa_20251113" 
# RUN_ID = "261PoLe_20251128" 
# RUN_ID = "284ChGe_20251030" 
# RUN_ID = "307BeJa_20251103" 
# RUN_ID = "314DuHu_20251009" 
# RUN_ID = "374SaSa_20251030" 
# RUN_ID = "394ZoMo_20251114" 
# RUN_ID = "402RaAx_20251128" 
# RUN_ID = "444BoAl_20251104" 
# RUN_ID = "459PoQu_20251020" 
# RUN_ID = "500TeJo_20251114" 
# RUN_ID = "592RiNa_20251028" 
# RUN_ID = "622PiSt_20251104" 
# RUN_ID = "684LuSh_20251013" 
# RUN_ID = "743MoGi_20251103" 
# RUN_ID = "797BoAm_20251021" 
# RUN_ID = "802BlHu_20251028" 
# RUN_ID = "923AoMo_20251017" 
# RUN_ID = "997FuLu_20251112" 
# RUN_ID = "890PeTh_20251023" 

########

# SET UP CACHE PATHS PARAMETERS
BASE_DERIVED = Path("data/derived") / RUN_ID
CACHE_DIR = BASE_DERIVED / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True) #create cache directory if it doesn't exist

# INDIVIDUAL CACHE PATH
CACHE_01_PARTICIPANTS = CACHE_DIR / "01_participants.parquet"

CACHE_02_MASTER = CACHE_DIR / "02a_master.parquet"
CACHE_02_EMG = CACHE_DIR / "02b_emg_compact.parquet"
CACHE_02_TORQUE = CACHE_DIR / "02c_torque_compact.parquet"
CACHE_02_METADATA = CACHE_DIR / "02d_metadata.parquet"

CACHE_03_SEQUENCE_PICKER = CACHE_DIR / "03_sequence_picked.json"

CACHE_04_VC_KNOBS_EVENTS = CACHE_DIR / "04_vc_knobs_events.parquet"


CACHE_05a_BIA_IMPORT = CACHE_DIR / "05a_bia_compact.parquet"
CACHE_05b_BIA_SYNC = CACHE_DIR / "05b_bia_sync.parquet"

CACHE_06a_NIRS_IMPORT = CACHE_DIR / "06a_nirs_compact.parquet"
CACHE_06b_NIRS_SYNC = CACHE_DIR / "06b_nirs_sync.parquet"


CACHE_07_MYOTON = CACHE_DIR / "07_myoton_compact.parquet"

## 1. PARTICIPANT INFO LOADER

In [ ]:
# 01. Participant info loader.  ACTIVE SUBJECT (single run per participant) ## fix & remove the deprecated run method
# ============================================================
#CHECK FOR CACHE PRESENT
if CACHE_01_PARTICIPANTS.exists():
    participants_df = pd.read_parquet(CACHE_01_PARTICIPANTS)
else:
    %run sources/01_participants.py #run the source script to create participants_df
    participants_df.to_parquet(CACHE_01_PARTICIPANTS, index=False) # save to cache for future runs
##



participant_id = RUN_ID.split("_", 1)[0]

participant_match = (participants_df["participant_id"] == participant_id)
assert participant_match.sum() == 1, (
    f"Expected exactly 1 match for participant_id={participant_id}, "
    f"got {participant_match.sum()}"
)

participant_row = participants_df.loc[participant_match].iloc[0]



## 2. EMG LOADER

In [ ]:
# 02 EMG IMPORT 

if CACHE_02_MASTER.exists():
    master_index_grid = pd.read_parquet(CACHE_02_MASTER)
    emg_compact_df = pd.read_parquet(CACHE_02_EMG)
    torque_compact_df = pd.read_parquet(CACHE_02_TORQUE)

    with open(CACHE_02_METADATA, "r", encoding="utf-8") as f:
        meta = json.load(f)
    ts_ref = pd.to_datetime(meta["ts_ref"])


else:
    local_namespace = runpy.run_path(
        "sources/02_emg_import.py",
        init_globals={
            "RUN_ID": RUN_ID,
            "RAW_ROOT": RAW_ROOT,
            "RAW_EMG_DIR": RAW_EMG_DIR,
            "participant_row": participant_row,
        },
    )

    master_index_grid = local_namespace["master_index_grid"]
    emg_compact_df = local_namespace["emg_compact_df"]
    torque_compact_df = local_namespace["torque_compact_df"]
    ts_ref = local_namespace["ts_ref"]

    del local_namespace
    gc.collect()

    master_index_grid.to_parquet(CACHE_02_MASTER, index=False)
    emg_compact_df.to_parquet(CACHE_02_EMG, index=False)
    torque_compact_df.to_parquet(CACHE_02_TORQUE, index=False)

    with open(CACHE_02_METADATA, "w", encoding="utf-8") as f:
        json.dump({"ts_ref": str(ts_ref)}, f)



## 03. LABELLING SEQUENCES

In [ ]:
# 03a. : MANUAL SEQUENCING THE RESEARCH PROTOCOLE USING EMG DATA (TORQUE)
assert SEQ_ORDER[0] == "INIT" and SEQ_ORDER[-1] == "END"
%matplotlib widget

assert "CACHE_DIR" in globals()
CACHE_03A_SEQUENCE_PICKER = CACHE_DIR / "03a_sequence_picked.json"

if not CACHE_03A_SEQUENCE_PICKER.exists():
    runpy.run_path(
        "sources/03a_sequence_picker.py",
        init_globals={
            "RUN_ID": RUN_ID,
            "CACHE_DIR": CACHE_DIR,
            "master_index_grid": master_index_grid,
            "torque_compact_df": torque_compact_df,
            "SEQ_ORDER": SEQ_ORDER,
            "ts_ref": ts_ref,
        },
    )
else:
    print("[SKIP] 03a_sequence_picked.json exists:", RUN_ID, " / cache /", CACHE_03A_SEQUENCE_PICKER.name)


In [ ]:
# 03b. Labeling
%matplotlib inline 
# revert back to inline for following steps

if "SEQ" in master_index_grid.columns :
    print("[SKIP] SEQ already present in master_index_grid.")
else:
    local_namespace = runpy.run_path(
        "sources/03b_sequence_labeling.py",
        init_globals={
            "RUN_ID": RUN_ID,
            "CACHE_DIR": CACHE_DIR,
            "master_index_grid": master_index_grid,
            "SEQ_ORDER": SEQ_ORDER,
            "emg_compact_df": emg_compact_df,
            "torque_compact_df": torque_compact_df,
        },
    )

    master_index_grid = local_namespace["master_index_grid"]

    #upgrade with seq columns
    master_index_grid.to_parquet(CACHE_02_MASTER, index=False)
    emg_compact_df.to_parquet(CACHE_02_EMG, index=False)
    torque_compact_df.to_parquet(CACHE_02_TORQUE, index=False)
    
    del local_namespace
    gc.collect()

    print("[DONE] 03b. Sequence labeling completed and master_index_grid updated with SEQ column.")


## 04. VOLUNTARY CONTRACTION DETECTION (VC)

In [ ]:
# 04a VC DETECTION — TUNE (knobs in notebook)

import runpy, gc
import pandas as pd

CACHE_04_VC_KNOBS_EVENTS = CACHE_DIR / "04_vc_knobs_events.parquet"

# Skip if cache already has events
skip = False
if CACHE_04_VC_KNOBS_EVENTS.exists():
    df = pd.read_parquet(CACHE_04_VC_KNOBS_EVENTS)
    if ("row_type" in df.columns) and df["row_type"].eq("event").any():
        print("[SKIP] 04 VC: cache already contains events.")
        skip = True

# knobs - EDIT THOSE AND RERUN CELL UNTIL VC ARE PERFECTLY DETECTED 
VC_KNOBS_BY_SEQ = {
        "WU":            {"thr_frac": 0.30, "merge_gap_s": 0.85, "min_dur_s": 3.00},
        "MVC_REF":       {"thr_frac": 0.40, "merge_gap_s": 0.10, "min_dur_s": 3.00},
        "SVC_REF":       {"thr_frac": 0.25, "merge_gap_s": 0.90, "min_dur_s": 3.00},
        "EX_DYN":        {"thr_frac": 0.25, "merge_gap_s": 0.25, "min_dur_s": 1.50},
        "MVC_RECOV_DYN": {"thr_frac": 0.40, "merge_gap_s": 0.10, "min_dur_s": 3.00},
        "EX_STA":        {"thr_frac": 0.25, "merge_gap_s": 0.65, "min_dur_s": 50.00},
        "MVC_RECOV_STA": {"thr_frac": 0.40, "merge_gap_s": 0.10, "min_dur_s": 3.00},
}

if not skip:
    local_namespace = runpy.run_path(
        "sources/04_vc_detection.py",
        init_globals={
            "RUN_ID": RUN_ID,
            "master_index_grid": master_index_grid,
            "emg_compact_df": emg_compact_df,
            "torque_compact_df": torque_compact_df,
            "CACHE_04_VC_KNOBS_EVENTS": CACHE_04_VC_KNOBS_EVENTS,
            "VC_KNOBS_BY_SEQ": VC_KNOBS_BY_SEQ,  # <- overrides for this run
        },
    )

    master_index_grid = local_namespace["master_index_grid_out"]
    vc_events_df = local_namespace["vc_events_df"]

    del local_namespace
    gc.collect()


In [ ]:
# 04b VC DETECTION — COMMIT (writes cache)

import runpy, gc

CACHE_04_VC_KNOBS_EVENTS = CACHE_DIR / "04_vc_knobs_events.parquet"

# Use the same VC_KNOBS_BY_SEQ currently in your notebook (optional)
VC_KNOBS_BY_SEQ = globals().get("VC_KNOBS_BY_SEQ", None)

local_namespace = runpy.run_path(
    "sources/04_vc_detection.py",
    init_globals={
        "RUN_ID": RUN_ID,
        "master_index_grid": master_index_grid,
        "torque_compact_df": torque_compact_df,
        "CACHE_04_VC_KNOBS_EVENTS": CACHE_04_VC_KNOBS_EVENTS,
        "VC_KNOBS_BY_SEQ": VC_KNOBS_BY_SEQ,
        "VC_COMMIT": True,
        "VC_PLOT": False,  # skip plotting during commit for speed
        "emg_compact_df": emg_compact_df, # pass the emg_compact_df for labeling in the source script
        "torque_compact_df": torque_compact_df, # pass the torque_compact_df for labeling in the source script
    },
)

master_index_grid = local_namespace["master_index_grid_out"]
emg_compact_df_out = local_namespace["emg_compact_df_out"]
torque_compact_df_out = local_namespace["torque_compact_df_out"]    
vc_events_df = local_namespace["vc_events_df"]

#upgrade now with VC columns
master_index_grid.to_parquet(CACHE_02_MASTER, index=False)
emg_compact_df_out.to_parquet(CACHE_02_EMG, index=False)
torque_compact_df_out.to_parquet(CACHE_02_TORQUE, index=False)

del local_namespace
gc.collect()

print("[OK] 04 VC committed.")


## 5. BIA LOADER & SYNC

In [ ]:
# 5a. BIA IMPORT

local_namespace = runpy.run_path(
    "sources/05a_bia_import.py",
    init_globals={
        "RUN_ID": RUN_ID,
        "ts_ref": ts_ref,
        "RAW_BIA_DIR": RAW_BIA_DIR,
        "CACHE_05a_BIA_IMPORT": CACHE_05a_BIA_IMPORT,
        "master_index_grid": master_index_grid,
    },
)

bia2_compact_df = local_namespace["bia2_compact_df_out"]
bia4_compact_df = local_namespace["bia4_compact_df_out"]
bia2_freqs_hz = local_namespace["bia2_freqs_hz_out"]
bia4_freqs_hz = local_namespace["bia4_freqs_hz_out"]

# Optional: keep the dict if you like, otherwise drop it
bia_tables = local_namespace["bia_tables_out"]

# Cleanup: remove transient variables (no globals sweep)
del local_namespace
gc.collect()


In [ ]:
# 5b. BIA SYNC

# knobs - EDIT THOSE AND RERUN CELL UNTIL SYNC LOOKS GOOD 
BIA_SYNC_ANCHOR_SEQ = "MVC_REF"
BIA_SYNC_TARGET_HZ  = 9760.0
BIA_SYNC_LAG_MIN_S  = -5.0
BIA_SYNC_LAG_MAX_S  =  5.0
BIA_SYNC_LAG_STEP_S =  0.02
BIA_SYNC_MANUAL_NUDGE_S = 0.0
BIA_SYNC_PLOT   = True
BIA_SYNC_COMMIT = False

CACHE_06_BIA_SYNC = CACHE_DIR / "05b_bia_sync.parquet"

local_namespace = runpy.run_path(
    "sources/05b_bia_sync.py",
    init_globals={
        "RUN_ID": RUN_ID,
        "master_index_grid": master_index_grid,

        "bia2_compact_df": bia2_compact_df,
        "bia4_compact_df": bia4_compact_df,
        "bia4_freqs_hz": bia4_freqs_hz,

        "torque_compact_df": torque_compact_df,
        "TORQUE_COL": "torque_raw",

        "CACHE_05b_BIA_SYNC": CACHE_06_BIA_SYNC,

        "BIA_SYNC_ANCHOR_SEQ": BIA_SYNC_ANCHOR_SEQ,
        "BIA_SYNC_TARGET_HZ": BIA_SYNC_TARGET_HZ,
        "BIA_SYNC_LAG_MIN_S": BIA_SYNC_LAG_MIN_S,
        "BIA_SYNC_LAG_MAX_S": BIA_SYNC_LAG_MAX_S,
        "BIA_SYNC_LAG_STEP_S": BIA_SYNC_LAG_STEP_S,
        "BIA_SYNC_MANUAL_NUDGE_S": BIA_SYNC_MANUAL_NUDGE_S,

        "BIA_SYNC_PLOT": BIA_SYNC_PLOT,
        "BIA_SYNC_COMMIT": BIA_SYNC_COMMIT,
    }
)

bia2_compact_aligned_df = local_namespace["bia2_compact_aligned_df_out"]
bia4_compact_aligned_df = local_namespace["bia4_compact_aligned_df_out"]


# after building the tune QC figure
bia_sync_qc_fig_out = local_namespace.get("bia_sync_qc_fig_out", None)
bia_sync_qc_fig_name = f"05b_bia_sync_tune_{RUN_ID}.png"


In [ ]:
# 5c. BIA SYNC — COMMIT (no recompute)

cache_dir = CACHE_06_BIA_SYNC.parent
cache_dir.mkdir(parents=True, exist_ok=True)

bia2_compact_aligned_df.to_parquet(cache_dir / "05b_bia2_compact_aligned.parquet", index=False)
bia4_compact_aligned_df.to_parquet(cache_dir / "05b_bia4_compact_aligned.parquet", index=False)



print("[05b_bia_sync] data committed.")

# Export PNG (only if fig exists)
if "bia_sync_qc_fig_out" in globals() and bia_sync_qc_fig_out is not None:
    qc_export_dir = Path("results/QC_EXPORT")
    qc_export_dir.mkdir(parents=True, exist_ok=True)

    png_path = qc_export_dir / f"05b_bia_sync_{RUN_ID}.png"
    bia_sync_qc_fig_out.savefig(png_path, dpi=300, bbox_inches="tight")
    print(f"[05b_bia_sync] PNG exported: {png_path}")
else:
    print("[05b_bia_sync] No QC figure found in memory. PNG not exported.")

_ = gc.collect()


## 6. NIRS LOADER & SYNC

In [ ]:
# 6a. NIRS IMPORT

import runpy
import gc
from pathlib import Path

# Cache handle (mirrors BIA)
CACHE_06a_NIRS_IMPORT = CACHE_DIR / "06a_nirs_import.parquet"

local_namespace = runpy.run_path(
    "sources/06a_nirs_import.py",
    init_globals={
        "RUN_ID": RUN_ID,
        "ts_ref": ts_ref,
        "RAW_NIRS_DIR": RAW_NIRS_DIR,           # Path to raw NIRS folder
        "CACHE_06a_NIRS_IMPORT": CACHE_06a_NIRS_IMPORT,
        "master_index_grid": master_index_grid,
    },
)


nirs_compact_df = local_namespace["nirs_compact_df_out"]

# Optional: keep dict output
nirs_tables = local_namespace.get("nirs_tables_out", None)

del local_namespace
garbage_collected = gc.collect() # clean the unreachable object and prevent notebook echo after gc

In [ ]:
# 6b. NIRS SYNC — TUNE

import runpy
import gc

# knobs - EDIT THOSE AND RERUN CELL UNTIL SYNC LOOKS GOOD
NIRS_SYNC_TARGET_SEQ = "MVC_REF"

# Pick ONE scoring column (must exist in nirs_compact_df)
# Recommended default:
NIRS_SYNC_SIGNAL_COL = "nirs_hbdiff_tx1"

NIRS_SYNC_LAG_MIN_S = -5
NIRS_SYNC_LAG_MAX_S = 5
NIRS_SYNC_STEP_S    =  0.05
NIRS_MANUAL_NUDGE_S = 0

NIRS_SYNC_PLOT   = True
NIRS_SYNC_COMMIT = False

CACHE_08_NIRS_SYNC = CACHE_DIR / "06b_nirs_sync.parquet"

local_namespace = runpy.run_path(
    "sources/06b_nirs_sync.py",
    init_globals={
        "RUN_ID": RUN_ID,
        "master_index_grid": master_index_grid,

        "nirs_compact_df": nirs_compact_df,

        # optional QC overlay (script checks existence)
        "torque_compact_df": torque_compact_df,
        "TORQUE_COL": "torque_raw",

        "CACHE_06b_NIRS_SYNC": CACHE_08_NIRS_SYNC,

        "NIRS_SYNC_TARGET_SEQ": NIRS_SYNC_TARGET_SEQ,
        "NIRS_SYNC_SIGNAL_COL": NIRS_SYNC_SIGNAL_COL,
        "NIRS_SYNC_LAG_MIN_S": NIRS_SYNC_LAG_MIN_S,
        "NIRS_SYNC_LAG_MAX_S": NIRS_SYNC_LAG_MAX_S,
        "NIRS_SYNC_STEP_S": NIRS_SYNC_STEP_S,
        "NIRS_MANUAL_NUDGE_S": NIRS_MANUAL_NUDGE_S,

        "NIRS_SYNC_PLOT": NIRS_SYNC_PLOT,
        "NIRS_SYNC_COMMIT": NIRS_SYNC_COMMIT,
    },
)

nirs_compact_aligned_df = local_namespace["nirs_compact_aligned_df_out"]

# after building the tune QC figure
nirs_sync_qc_fig_out = local_namespace.get("nirs_sync_qc_fig_out", None)
nirs_sync_qc_fig_name = f"06b_nirs_sync_tune_{RUN_ID}.png"

del local_namespace
gc.collect()

In [ ]:
# 6c. NIRS SYNC — COMMIT (no recompute)

cache_dir = CACHE_08_NIRS_SYNC.parent
cache_dir.mkdir(parents=True, exist_ok=True)

nirs_compact_aligned_df.to_parquet(cache_dir / "06b_nirs_compact_aligned.parquet", index=False)



# export the tune figure 
assert "nirs_sync_qc_fig_out" in globals(), "Missing nirs_sync_qc_fig_out. Run the TUNE cell (plot) before COMMIT."
assert "nirs_sync_qc_fig_name" in globals(), "Missing nirs_sync_qc_fig_name. Define it in the TUNE cell."

qc_export_dir = Path("results/QC_EXPORT")
qc_export_dir.mkdir(parents=True, exist_ok=True)

png_path = qc_export_dir / nirs_sync_qc_fig_name
nirs_sync_qc_fig_out.savefig(png_path, dpi=300, bbox_inches="tight")
plt.close(nirs_sync_qc_fig_out)

print(f"Step 06b committed. PNG exported: {png_path}")

print("Step 06b committed.")



## 7. MYOTON LOADER & SYNC

In [ ]:
# 7. MYOTON IMPORT (cache-aware) + COMMIT (same cell)

if CACHE_07_MYOTON.exists():
    myoton_compact_df = pd.read_parquet(CACHE_07_MYOTON)
else:
    local_namespace = runpy.run_path(
        "sources/07_myoton_import_sync.py",
        init_globals={
            "RUN_ID": RUN_ID,
            "RAW_MYOTON_DIR": RAW_MYOTON_DIR,
            "ts_ref": ts_ref,
            "master_index_grid": master_index_grid,
            "torque_compact_df": torque_compact_df,
        }
    )

    myoton_compact_df = local_namespace["myoton_compact_df_out"]

    # COMMIT (same cell is fine here)
    myoton_compact_df.to_parquet(CACHE_07_MYOTON, index=False)

print("MYOTON rows:", len(myoton_compact_df))
print("MYOTON cache:", CACHE_07_MYOTON, "| exists:", CACHE_07_MYOTON.exists())

## 8. FINAL FULL QC PLOTS

In [ ]:
# 8. FINAL QUALITY CHECK — PLOT ONLY (runpy)
%matplotlib inline


local_namespace = runpy.run_path(
    "sources/08_final_quality_check.py",
    init_globals={
        "RUN_ID": RUN_ID,
        "CACHE_02_EMG" : CACHE_02_EMG,
        "CACHE_02_TORQUE": CACHE_02_TORQUE,
        "CACHE_05b_BIA_SYNC": CACHE_05b_BIA_SYNC,
        "CACHE_06b_NIRS_SYNC": CACHE_06b_NIRS_SYNC,
        "CACHE_DIR": CACHE_DIR,

        "master_index_grid": master_index_grid,

        "bia2_compact_aligned_df": bia2_compact_aligned_df,
        "bia4_compact_aligned_df": bia4_compact_aligned_df,
        
        "nirs_compact_aligned_df": nirs_compact_aligned_df,
        "CACHE_07_MYOTON": CACHE_07_MYOTON,
    },
)

# OUTPUTS 
qc_figs = local_namespace["qc_figs_out"]  # list of (fig, name)

for fig, name in qc_figs:
    display(fig)
del local_namespace
gc.collect()

## 9. DATA EXPORT

In [ ]:
# 9a. QC EXPORT (PDF)

from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages

QC_EXPORT_ROOT = Path("results") / "QC_EXPORT"
QC_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

qc_pdf_path = QC_EXPORT_ROOT / f"QC_PLOT_EXPORT_{RUN_ID}.pdf"

with PdfPages(qc_pdf_path) as pdf:
    for fig_obj, fig_name in qc_figs:
        pdf.savefig(fig_obj)

print(f"[QC EXPORT] PDF created → {qc_pdf_path.resolve()}")

# DATA EXPORT
import shutil


DATA_EXPORT_ROOT = Path("results") / "DATA_EXPORT" / RUN_ID
DATA_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

# 9b. DATA EXPORT (PARQUET)
PARQUET_EXPORT_LIST = [
    "01_participants.parquet",
    "02b_emg_compact.parquet",
    "02a_master.parquet",
    "02c_torque_compact.parquet",
    # "05_bia2_compact.parquet",
    "05a_bia2_freqs_hz.parquet",
    # "05_bia4_compact.parquet",
    "05a_bia4_freqs_hz.parquet",
    "05b_bia2_compact_aligned.parquet",
    "05b_bia4_compact_aligned.parquet",
    "06b_nirs_compact_aligned.parquet",
    "07_myoton_compact.parquet",
    #"02_metadata.parquet",
    #"04_vc_knobs_events.parquet",
]

## also print the columns inside the parquet selected for export, for quick reference
print("Parquet files selected for export and their columns:")
for fname in PARQUET_EXPORT_LIST:
    src = CACHE_DIR / fname
    if src.exists():
        df = pd.read_parquet(src)
        print(f"- {fname}: columns = {list(df.columns)}")
    else:
        print(f"- {fname}: NOT FOUND in cache, will fail export.")
        
# --------------------------------------------------
# COPY
# --------------------------------------------------
assert isinstance(CACHE_DIR, Path), "CACHE_DIR must be a pathlib.Path"
assert CACHE_DIR.exists(), f"CACHE_DIR not found: {CACHE_DIR}"

copied = 0
for fname in PARQUET_EXPORT_LIST:
    src = CACHE_DIR / fname
    assert src.exists(), f"Missing export file: {src}"

    dst = DATA_EXPORT_ROOT / fname
    shutil.copy2(src, dst)

    print(f"[DATA EXPORT] Copied → {dst}")
    copied += 1

assert copied > 0, "Nothing copied (PARQUET_EXPORT_LIST empty?)"
print(f"\n[DATA EXPORT COMPLETE] → {DATA_EXPORT_ROOT.resolve()}")

